# Overview

## Target

Binary classification.

## Description
Data Science London is hosting a meetup on Scikit-learn.  This competition is a practice ground for trying, sharing, and creating examples of sklearn's classification abilities (if this turns in to something useful, we can follow it up with regression, or more complex classification problems).

We encourage participants to post code via the "Tutorials" link on the left.  Don't worry about accuracy or whether your code is perfect.  The aim here is to explore sklearn by using it. You do not need to use sklearn to enter the competition. If you're new, we hope you'll use this oppurtunity to practice a new tool.  If you're an expert, we hope you'll share the knowledge and document interesting ways to approach this problem.

![sklearn image](https://storage.googleapis.com/kaggle-media/competitions/kaggle/3428/media/scikit-learn-logo.png)

Scikit-learn (sklearn) is an established, open-source machine learning library, written in Python with the help of NumPy, SciPy and Cython.

Scikit-learn is very user friendly, has a consistent API, and provides extensive documentation. Its implementation is high quality due to strict coding standards and high test coverage.  Behind sklearn is a very active community, which is steadily improving the library.

## Evaluation

This is a binary classification task, You are evaluated on classification accuracy (the percentage of labels you predict correctly).  The training set has 1000 samples and the testing set has 9000.  Your prediction should be a 9000 x 1 vector of ones or zeros. You also need an Id column (1 to 9000) and should include a header. The format looks like this:

Id,Solution
1,0
2,1
3,1
...
9000,0

## Dataset

This is a synthetic data set of 40 features, representing objects from two classes (labeled as 0 or 1). The training set has 1000 samples and the testing set has 9000.

## Content
1. [Data Understanding](#1) 
1. [Data Exploration](#2)                          
1. [Feature Engineering](#3)
1. [Data Preprocessing](#4)
1. [Basic Model Building](#5)
1. [Model Tuning](#6)
1. [Submission](#7)
1. [Result](#8)

<a id="1"></a>
# 1) Data Understanding

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
from sklearn.metrics import make_scorer, f1_score, jaccard_score, accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler, Normalizer, MinMaxScaler
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split, cross_validate, cross_val_score
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier 
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.mixture import GaussianMixture
import math

import warnings
warnings.filterwarnings("ignore")  # Ignore all warnings (not recommended for debugging)

try:
    # Code that produces a specific warning
    pass
except SomeSpecificWarning:
    warnings.warn("Handling specific warning")

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

### Loading Datasets

In [ ]:
train_path = "/kaggle/input/data-science-london-scikit-learn/train.csv"
test_path = "/kaggle/input/data-science-london-scikit-learn/test.csv"
train_label_path = "/kaggle/input/data-science-london-scikit-learn/trainLabels.csv"

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
train_label = pd.read_csv(train_label_path)

In [ ]:
print(f'Train set shape : {train.shape} Test set shape : {test.shape} Label set shape : {train_label.shape}')

In [ ]:
train.head()

<a id="2"></a>
# 2) Data Exploration

### Adding Column Names

In [ ]:
SIZE = train.shape[1]

column_names = [f'column_{i}' for i in range(SIZE)]

# Load training data with column names
df_train = pd.read_csv(train_path, names=column_names)

# Load test data with column names
df_test = pd.read_csv(test_path, names=column_names)

# Load training labels with column name (assuming a single column for labels)
label_column_name = ["label"]
df_train_label = pd.read_csv(train_label_path, names=label_column_name)

* Train data: does not have missing value, all values are float64. Shape is 1000 rows and 40 columns. 
* Test data: does not have missing value, all values are float64. Shape is 9000 rows and 40 columns. 

In [ ]:
df_train.describe()

### Plots of Values

In [ ]:
# Calculate the number of rows and columns for the grid layout
num_features = len(df_train.columns)
num_cols = 5  # Number of columns in the grid
num_rows = math.ceil(num_features / num_cols)  # Number of rows in the grid

# Set the size of each subplot
plt.figure(figsize=(15, 5*num_rows))

# Plot histograms for all numeric variables in a grid layout
for i, col in enumerate(df_train.columns):
    plt.subplot(num_rows, num_cols, i+1)
    sns.histplot(df_train[col], kde=True)
    plt.title(f"{col} Histogram Plot")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.xticks(rotation=90)  # Rotate x-axis labels by 90 degrees for better readability

# Adjust layout
plt.tight_layout()
plt.show()


<a id="3"></a>
# 3) Feature Engineering

### Correlation Heatmap

In [ ]:
# Create a correlation matrix for df_train
correlation_matrix_train = df_train.corr()

# Create a heatmap for the correlation matrix of df_train with masking values > +0.5 and < -0.5
plt.figure(figsize=(24, 24))
plt.subplot(2, 1, 1)  # 2 rows, 1 column, first subplot
sns.heatmap(correlation_matrix_train, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix (df_train)')

# Create a correlation matrix for df_test
correlation_matrix_test = df_test.corr()

# Create a heatmap for the correlation matrix of df_test with masking values > +0.5 and < -0.5
plt.subplot(2, 1, 2)  # 2 rows, 1 column, second subplot
sns.heatmap(correlation_matrix_test, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix (df_test)')

plt.tight_layout()
plt.show()

Correlation measures the statistical relationship between two variables. In the context of feature selection or feature engineering, understanding the correlation between features (variables) is crucial. Here's what high and low correlation mean:

### High Correlation:

1. **Positive High Correlation:** Two variables are positively correlated if they tend to increase or decrease together. A high positive correlation means that when one variable goes up, the other variable also tends to go up, and vice versa.

2. **Negative High Correlation:** Two variables are negatively correlated if one variable tends to increase when the other variable decreases. A high negative correlation means that when one variable goes up, the other variable tends to go down, and vice versa.

In either case, a high correlation (close to +1 or -1) indicates a strong relationship between the variables. For example, if there is a high positive correlation between hours of study and exam scores, it means that students who study more tend to score higher in exams.

### Low Correlation:

1. **Low Positive Correlation:** Two variables have a low positive correlation if there is a weak tendency for them to increase or decrease together. A low positive correlation (close to 0) suggests a weak relationship between the variables.

2. **Low Negative Correlation:** Two variables have a low negative correlation if there is a weak tendency for one variable to increase when the other decreases, and vice versa. A low negative correlation (close to 0) indicates a weak inverse relationship.

Low correlations imply that changes in one variable are not well-predicted by changes in the other variable. In feature selection, having low correlation between features is desirable, especially if the features are redundant. Redundant features (highly correlated) might not add significant information to the model, and removing them can simplify the model without losing much predictive power.

**Note:** The exact threshold for what constitutes high or low correlation can vary based on the context and the specific problem you are working on. Different fields might have different conventions regarding what is considered a significant correlation.

### Balanced Labels

Having balanced labels is important in machine learning, especially in classification tasks, because it ensures that the learning algorithm is exposed to an equal representation of all classes. When classes are imbalanced (one or more classes have significantly more or fewer instances than others), the machine learning model might be biased towards the majority class, leading to poor performance, especially for the minority class.

To address this issue, techniques like resampling (oversampling the minority class or undersampling the majority class) or using specific algorithms designed to handle imbalanced datasets can be employed to create a balanced representation of classes. These techniques help the model learn the patterns from all classes equally, leading to a more reliable and accurate predictive model, especially for the under-represented classes.

In [ ]:
df_train_label.value_counts()

### Missing Values

In [ ]:
# Missing Value Table
def missing_value_table(df):
    # Calculate the number of missing values for each column
    missing_value = df.isna().sum().sort_values(ascending=False)
    
    # Calculate the percentage of missing values for each column
    missing_value_percent = 100 * missing_value / len(df)
    
    # Create a DataFrame to store missing values and percentages
    missing_value_table = pd.concat([missing_value, missing_value_percent], axis=1)
    missing_value_table = missing_value_table.rename(columns={0: 'Missing Values', 1: '% Value'})
    missing_value_table = missing_value_table[missing_value_table['Missing Values'] > 0]
    # Apply background gradient to the missing value table for visualization
    cm = sns.light_palette("lightgreen", as_cmap=True)
    missing_value_table_return = missing_value_table.style.background_gradient(cmap=cm)

    return missing_value_table_return

In [ ]:
print("Train Dataset Missing Value Table: ")
missing_value_table(df_train)

In [ ]:
print("Test Dataset Missing Value Table: ")
missing_value_table(df_test)

* Train and test sets do not contain missing values.

<a id="4"></a>
# 4) Data Processing

### X and y Variables

In [ ]:
X = df_train.copy()
y = df_train_label.copy()
y = y.values.ravel()

### Train, Test Split

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### Principal Component Analysis (PCA)

If your dataset is large enough, it's generally recommended to perform PCA after the train-test split to prevent data leakage and ensure a more realistic evaluation of your model. However, if your dataset is small and you want to capture the maximum variability, you might consider applying PCA before splitting. Always be cautious about data leakage and validate your approach thoroughly to ensure the integrity of your model evaluation.

In [ ]:
# Feature Engineering: PCA for Dimensionality Reduction
pca = PCA(n_components=0.95, random_state=42)

In [ ]:
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

### Feature Scaling

Feature scaling is a technique used in machine learning to standardize the range of independent variables or features in the data. It's a critical step in many machine learning algorithms, particularly those that are distance-based (e.g., k-nearest neighbors, support vector machines) or gradient descent-based (e.g., linear regression, logistic regression, neural networks).

The goal of feature scaling is to bring all features to a similar scale, preventing some features from dominating others. Here are a few common methods of feature scaling:

1. Min-Max Scaling (Normalization):

Method: Scales features to a fixed range, typically [0, 1].
How: It transforms the data so that the minimum value of the feature is mapped to 0, the maximum value is mapped to 1, and every other value is mapped to a decimal between 0 and 1.

2. Standardization (Z-score Normalization):

Method: Scales features so they follow a standard normal distribution with a mean (μ) of 0 and a standard deviation (σ) of 1.
How: It transforms the data so that the mean of the feature values is 0, and their variance is 1. Features values are centered around the mean.

3. Robust Scaling:

Method: Scales features using the median and interquartile range (IQR) instead of mean and standard deviation. It is robust against outliers.
How: It scales the feature values by subtracting the median and dividing by the interquartile range.

Feature scaling ensures that each feature contributes approximately proportionately to the final distance or error metric used by the machine learning algorithm. It helps in improving the convergence speed and accuracy of many machine learning algorithms, especially those based on distances or gradients.

4. Normalizer:

is a feature scaling technique used in machine learning to scale individual samples to have unit norm. This means that after applying normalizer, each sample (i.e., each row in the dataset) will have a Euclidean norm (L2 norm) of 1.

How Normalizer Works:
When normalizer is applied, it operates on the rows (samples) of the dataset. For each row, the values are scaled in such a way that the Euclidean norm of the row becomes 1. 

The Euclidean norm of a vector X is calculated as: euclidean_norm = np.linalg.norm(vector)

In [ ]:
# Standardize features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_pca)
X_test_scaled = scaler.transform(X_test_pca)

<a id="5"></a>
# 5) Basic Model Building

In [ ]:
# KNN Classifier
knn_classifier = KNeighborsClassifier(n_neighbors=5)
knn_classifier.fit(X_train_scaled, y_train)
knn_predictions = knn_classifier.predict(X_test_scaled)

# Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_scaled, y_train)
rf_predictions = rf_classifier.predict(X_test_scaled)

# LGBM Classifier
lgbm_classifier = LGBMClassifier(n_estimators=100, random_state=42)
lgbm_classifier.fit(X_train_scaled, y_train)
lgbm_predictions = lgbm_classifier.predict(X_test_scaled)

# XGBoost Classifier
xgb_classifier = XGBClassifier(n_estimators=100, random_state=42)
xgb_classifier.fit(X_train_scaled, y_train)
xgb_predictions = xgb_classifier.predict(X_test_scaled)

# Support Vector Classifier (SVC)
svc_classifier = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svc_classifier.fit(X_train_scaled, y_train)
svc_predictions = svc_classifier.predict(X_test_scaled)

# Evaluation Metrics
models = ['KNN', 'Random Forest', 'LGBM', 'XGBoost', 'SVC']
predictions = [knn_predictions, rf_predictions, lgbm_predictions, xgb_predictions, svc_predictions]

for model, preds in zip(models, predictions):
    accuracy = accuracy_score(y_test, preds)
    report = classification_report(y_test, preds)
    matrix = confusion_matrix(y_test, preds)
    
    print(f"Model: {model}")
    print(f"Accuracy: {accuracy:.2f}")
    print("Classification Report:\n", report)
    print("Confusion Matrix:\n", matrix)
    print("-" * 50)

### Plots of Results

In [ ]:
# Initialize lists to store metrics and models
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
models = ['KNN', 'Random Forest', 'LGBM', 'XGBoost', 'SVC']

# Calculate metrics and store them in the respective lists
for preds in predictions:
    accuracy = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted')
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

# Rest of your plotting code...

# Plot accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(models, accuracy_scores, marker='o', label='Accuracy')
plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.title('Accuracy Scores for Different Models')
plt.legend()

# Plot precision, recall, and F1-score
plt.subplot(1, 2, 2)
plt.plot(models, precision_scores, marker='o', label='Precision')
plt.plot(models, recall_scores, marker='o', label='Recall')
plt.plot(models, f1_scores, marker='o', label='F1-Score')
plt.xlabel('Models')
plt.ylabel('Scores')
plt.title('Precision, Recall, and F1-Score for Different Models')
plt.legend()

plt.tight_layout()
plt.show()

### Gaussian Mixture

Gaussian Mixture Model (GMM) is a probabilistic model that represents a mixture of several Gaussian (normal) distributions. Each component in the mixture represents a cluster, and the model aims to capture the underlying patterns in the data by estimating the parameters of these Gaussian distributions.

Here are the key components of a Gaussian Mixture Model:

1. **Gaussian Distribution:** A Gaussian distribution (also known as normal distribution) is defined by its mean (\(\mu\)) and standard deviation (\(\sigma\)). It is a bell-shaped curve characterized by these parameters.

2. **Mixture:** A GMM is a mixture of multiple Gaussian distributions. Each component of the mixture represents a cluster in the data. The GMM estimates the parameters (mean and covariance) of these Gaussian components.

3. **Parameters:** For each Gaussian component, the GMM estimates parameters such as mean (\(\mu\)), covariance (\(\Sigma\)), and weight (\(\pi\)). The weight (\(\pi\)) represents the proportion of data points belonging to each component.

4. **Probability Density Function (PDF):** The PDF of a GMM calculates the likelihood of a data point belonging to the entire mixture model. It is calculated as the weighted sum of the PDFs of individual Gaussian components.

5. **Expectation-Maximization (EM) Algorithm:** GMM parameters are typically estimated using the EM algorithm. The EM algorithm iteratively optimizes the parameters by performing an expectation step (E-step) to calculate the probability of each data point belonging to each component and a maximization step (M-step) to update the parameters based on these probabilities.

GMMs are commonly used in various applications, including clustering, density estimation, and anomaly detection. They are especially useful when the underlying data is assumed to be generated from multiple Gaussian distributions, and the goal is to uncover these underlying patterns.

In [ ]:
x_all = np.r_[df_train,df_test]
print('x_all shape :', x_all.shape)

In [ ]:
# Define the parameter grid to search
param_grid = {
    'n_components': [3, 4, 5],
    'covariance_type': ['spherical', 'tied', 'diag', 'full']
}

# Create a GMM instance
gmm = GaussianMixture(random_state=42)

# Create GridSearchCV object
grid_search = GridSearchCV(estimator=gmm, param_grid=param_grid, cv=5)

# Fit the GridSearchCV to your data
grid_search.fit(x_all)

# Get the best GMM model from the grid search
best_gmm = grid_search.best_estimator_

# Fit the best GMM model to your data
best_gmm.fit(x_all)

# Transform your data using the best GMM model
gmm_train = best_gmm.predict_proba(df_train)
gmm_test = best_gmm.predict_proba(df_test)

<a id="6"></a>
# 6) Model Tuning

### GridSearch

In [ ]:
# Define parameter grids and scalers for each model
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier

models = {
    'Random Forest': {
        'model': RandomForestClassifier(),
        'params': {
            'model__n_estimators': [10, 50, 100],
            'model__max_depth': [None, 5, 10],
        }
    },
    'LGBMClassifier': {
        'model': LGBMClassifier(),
        'params': {
            'model__n_estimators': [10, 50, 100],
            'model__max_depth': [None, 5, 10],
        }
    },
    'Logistic Regression': {
        'model': LogisticRegression(),
        'params': {
            'model__C': [0.1, 1, 10],
        }
    },
    'Gradient Boosting Classifier': {
        'model': GradientBoostingClassifier(),
        'params': {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 6],
        }
    },
    'Naive Bayes': {
        'model': GaussianNB(),
        'params': {}
    },
    'K-Nearest Neighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'model__n_neighbors': [3, 5, 7, 9],
        }
    }
}

scalers = [StandardScaler(), RobustScaler(), MinMaxScaler(), Normalizer()]

best_models = {}
best_scores = {}

# Define the evaluation metrics
scoring = {
    'Accuracy': make_scorer(accuracy_score),
    'Precision': make_scorer(precision_score, average='weighted'),
    'Recall': make_scorer(recall_score, average='weighted'),
    'F1 Score': make_scorer(f1_score, average='weighted')
}

# Iterate through each model
for model_name, model_config in models.items():
    best_model = None
    best_score = 0
    
    for scaler in scalers:
        model_pipeline = Pipeline(steps=[
            ('scaler', scaler),
            ('model', model_config['model'])
        ])

        grid_search = GridSearchCV(model_pipeline, model_config['params'], cv=10, scoring=scoring, n_jobs=-1, refit='F1 Score')
        grid_search.fit(gmm_train, y)
        
        # Print the best parameters and best scores for the current model
        print(f"Model: {model_name}, Scaler: {scaler}")
        print(f"Best parameters: {grid_search.best_params_}")
        print(f"Best Cross-Validation Accuracy: {grid_search.cv_results_['mean_test_Accuracy'][grid_search.best_index_]:.2f}")
        print(f"Best Cross-Validation Precision: {grid_search.cv_results_['mean_test_Precision'][grid_search.best_index_]:.2f}")
        print(f"Best Cross-Validation Recall: {grid_search.cv_results_['mean_test_Recall'][grid_search.best_index_]:.2f}")
        print(f"Best Cross-Validation F1 Score: {grid_search.cv_results_['mean_test_F1 Score'][grid_search.best_index_]:.2f}\n")
        
        # Check if this model is the best so far
        if grid_search.cv_results_['mean_test_F1 Score'][grid_search.best_index_] > best_score:
            best_score = grid_search.cv_results_['mean_test_F1 Score'][grid_search.best_index_]
            best_model = grid_search.best_estimator_
    
    best_models[model_name] = best_model
    best_scores[model_name] = best_score

### Stacking

Stacking is an ensemble learning technique that combines multiple base models to create a new model, often called a meta-learner or blender. Stacking works by training several base models on the training data, then using these base models to make predictions. These predictions are then used as features to train the meta-learner, which makes the final predictions.

In [ ]:
# Create a list of (name, model) tuples for the StackingClassifier
estimators = []
for model_name, best_model in best_models.items():
    estimators.append((model_name, best_model))

# Create the StackingClassifier with LogisticRegression as the final estimator
stacking_clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())

# Define the parameter grid for hyperparameter tuning of the StackingClassifier
param_grid = {
    'final_estimator__C': [0.1, 1, 10],  # Example hyperparameters for the LogisticRegression final estimator
    'stack_method': ['auto', 'predict_proba']
}

# Create the GridSearchCV object for hyperparameter tuning
grid_search = GridSearchCV(estimator=stacking_clf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Perform hyperparameter tuning on the meta-learner
grid_search.fit(gmm_train, y)

# Get the best meta-learner
best_meta_learner = grid_search.best_estimator_

# Predictions on the test data using the optimized StackingClassifier
stacking_predictions = best_meta_learner.predict(gmm_test)


<a id="7"></a>
# 7) Submission

In [ ]:
# Create a DataFrame for submission
df_submission = pd.DataFrame({
    'Id': range(1, len(stacking_predictions) + 1),  # Assuming Id starts from 1
    'Solution': stacking_predictions
})

# Save the predictions to a CSV file
df_submission.to_csv('submission.csv', index=False)

<a id="8"></a>
# 8) Result

* Highest Private score: 0.99192
* Highest Public score: 0.9918